In [ ]:
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import recall_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from DREAMwalk.Evaluation_index import DiseaseDrugRecall
from DREAMwalk.data_process import prepare_negative_samples
from DREAMwalk.predict_associations_id_LR import predict_dda, predict_drug_disease
from DREAMwalk.predict_associations_id_LR import MLP_predict_dda, SVM_predict_dda, RF_predict_dda, KNN_predict_dda, GBM_predict_dda

In [ ]:

def prediction_average_xboost(path_save):
    all_results = []
    for i in range(0,10):
        embeddingf='/home/yin/DREAMwalk-main/DREAMwalk-main/data/data_jiaqi_stitch_cutoff_merge/{}/embedding_file_yin.pkl'.format(path_save)
        disease_all_label_f = '/home/yin/DREAMwalk-main/DREAMwalk-main/demo/LiuRui/result_id/disease_label_all_id_{}.txt'.format(i)
        disease_drug_label_f = '/home/yin/DREAMwalk-main/DREAMwalk-main/demo/LiuRui/result_id/disease_label_drug_id_{}.txt'.format(i)
        disease_herb_label_f = '/home/yin/DREAMwalk-main/DREAMwalk-main/demo/LiuRui/result_id/disease_label_herb_id_{}.txt'.format(i)       
        ###  Predict drug-disease  association
        result_list = []
        disease_label_files = {'all': disease_all_label_f,'drug': disease_drug_label_f,'herb': disease_herb_label_f}
        for type in ['all', 'drug', 'herb']:
            set = type.upper()[0:4]
            modelf='/home/yin/DREAMwalk-main/DREAMwalk-main/demo/LiuRui/Repositioning_modelf/{}/clf_cutoff_negative/clf_{}_{}.pkl'.format(path_save, type, i)
            pairf=disease_label_files[type]
       
            Df = predict_dda(embeddingf=embeddingf, pairf=pairf, modelf=modelf, seed=20)
            true_filepath = '/home/yin/DREAMwalk-main/DREAMwalk-main/demo/LiuRui/result_id/disease_label_{}_id_{}.txt'.format(type,i)
            recall_calculator = DiseaseDrugRecall(Df, true_filepath)
            average_recall, recall_at_k, precision_at_k, f1_scores_at_k = recall_calculator.calculate_metrics(2)
           
            one_result = [set,average_recall, recall_at_k, precision_at_k, f1_scores_at_k] 
            result_list.append(one_result)  
        result_pd = pd.DataFrame(result_list, columns = ['set','average_recall','recall_at_k','precision_at_k','f1_scores_at_k'])    
        all_results.append(result_pd)
        save_path = '/home/yin/DREAMwalk-main/DREAMwalk-main/demo/LiuRui/Evaluate_Repositioning_parameters/no_vaild/XGB_no_vaild/test_{}_{}.csv'.format(path_save,i)
        result_pd.to_csv(save_path, index=False)
    return all_results

compare_parameter_pd_list = []
predict_res_list=[]
path = []
for t in ['T1', 'T2','T3']:
    for n in ['N1', 'N2','N3']:
        for w in ['W1', 'W2','W3']:
            path_save = t+n+w
            path.append(path_save)
for pathsave in path:
    all_results  = prediction_average_xboost(pathsave)